In [ ]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.send_queue_stage_writer import (
    build_sensor_messages_send_queue,
    validate_sensor_messages_send_queue,
)


In [ ]:


engine = get_engine_from_env()


In [ ]:
SCHEMA = str("capstone")
DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")


IF_EXISTS_FLAG = str("replace")

QUEUE_STATUS_DEFAULT = str("pending")
CHUNK_SIZE = int(10000)

SEND_QUEUE_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_timestamped_stage")
SEND_QUEUE_DESTINATION_TABLE_NAME = str("synthetic_sensor_messages_send_queue")


In [ ]:
send_queue_table_name = build_sensor_messages_send_queue(
    engine=engine,
    schema=SCHEMA,
    source_table=SEND_QUEUE_SOURCE_TABLE_NAME,
    target_table=SEND_QUEUE_DESTINATION_TABLE_NAME,
    if_exists=IF_EXISTS_FLAG,
    queue_status_default=QUEUE_STATUS_DEFAULT,
    chunk_size=CHUNK_SIZE,
)

print("Built table:", send_queue_table_name)

In [ ]:

validation_dataframe = validate_sensor_messages_send_queue(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_DESTINATION_TABLE_NAME,
)


In [ ]:

display(validation_dataframe)

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        message_key,
        observation_index,
        observation_timestamp,
        message_sequence_index,
        sensor_name,
        sensor_index,
        sensor_value,
        queue_status,
        queued_at,
        producer_sent_at
    FROM capstone.synthetic_sensor_messages_send_queue
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 104
    """
)


In [ ]:

display(sample_dataframe)

In [ ]:
pending_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT *
    FROM capstone.synthetic_sensor_messages_send_queue
    WHERE queue_status = 'pending'
      AND producer_sent_at IS NULL
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 500
    """
)

display(pending_dataframe.head(100))